In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('..')
%load_ext autoreload
%autoreload 2

In [ ]:
from soilgasflux_fcs import multiprocess_raw_data, json_reader, fcs
from synthetic import synthetic_postProcessing
import pathlib
import matplotlib.pyplot as plt
import xarray as xr

In [ ]:
data_path = pathlib.Path(r'M:\alex\00-SoilGasFlux_rawdata\temp4\test03')
a = json_reader.Initializer(folderPath=data_path)
df = a.prepare_rawdata()

In [ ]:
a = multiprocess_raw_data.Multiprocessor()
b = a.run(df=df, chamber_id='synthetic_test03')

In [ ]:
c = multiprocess_raw_data.Multiprocessor()
d = c.run_MC(df=df, chamber_id='synthetic_test03_1000log1_mc')

In [ ]:
expected_synthetic = synthetic_postProcessing.Synthetic(processed_data='./output/synthetic_test03_2025-03-31.nc',
                                                      raw_dataFolder=data_path)
expected_results = expected_synthetic.get_expectedResults()

In [ ]:
expected_results.to_netcdf('./output/expected_synthetic_test03_2025-03-31.nc')

In [ ]:
#ds = xr.open_dataset('./output/synthetic_test03_2025-03-31.nc')
#ds = xr.open_dataset('./output/expected_synthetic_test04_01_2025-03-31.nc')
ds = xr.open_dataset('./output/synthetic_test03_1000log1_mc_2025-03-31.nc')

In [ ]:
ds

In [ ]:
(-ds.isel(time=0, deadband=4, cutoff=10)['logprob(HM)']).plot()
plt.yscale('log')
#plt.ylim(-10000000,0)

In [ ]:
import numpy as np

In [ ]:
time=1

In [ ]:
u_range = (ds.isel(time=time).quantile(0.9, skipna=True, dim=['MC'])['dcdt(HM)']-ds.isel(time=time).quantile(0.1, skipna=True, dim=['MC'])['dcdt(HM)']).values.flatten()

In [ ]:
logprob = ds.isel(time=time)['logprob(HM)'].median(dim=['MC']).values.flatten()

In [ ]:
plt.scatter(u_range, -logprob)
plt.yscale('log')

In [ ]:
ds.isel(time=0)['logprob(HM)'].median(dim=['MC']).plot()

In [ ]:
ds.isel(time=15)['nRMSE(HM)'].plot()

In [ ]:
ds.isel(time=0)['RMSE(HM)'].plot()

In [ ]:
ds.isel(time=30)['dcdt(HM)'].plot()